# islide — M5 rendering-feel demo

M5 is **feel**, not features: no new traits, no wire-format change (the
`tiles` / `tile_geo` contract is untouched, module version stays 2.0.0).
Two things change when you interact:

1. **Center-first chunked reads** (Python). A render's read plan is split
   into 1024-px blocks (4×4 tiles), each fetched independently and ordered
   by distance to the viewport center. For a multi-block view the center
   block is pushed *first* and the full tile set follows as a second push;
   a view that fits one block renders exactly as before (one read, one
   push, byte-identical `tiles` / `tile_geo`).
2. **Level cross-fade** (JS only). When the selected pyramid level changes
   between pushes, the old level fades out over 300 ms
   (`frontend/blend.js`, `BLEND_MS`) while the new level draws at full
   opacity. The view accumulates tile geometry across pushes (`_tileGeo`),
   so the second push of a two-stage render never blanks the canvas, and
   the outgoing level stays visible until its alpha reaches zero.


## 1. Open the slide

The slide opens on a background thread; `wait()` blocks until it's ready.
Everything below uses the public API only — M5 added no new Python or
widget surface (the chunking lives inside the existing render pipeline).

In [ ]:
from islide import SlideViewer

# Test slide (data/ is gitignored): download data/CMU-1.tiff from
# https://openslide.cs.cmu.edu/download/openslide-testdata/Generic-TIFF/CMU-1.tiff
v = SlideViewer("data/CMU-1.tiff")
v.wait()
meta = v.backend.meta
print(f"{meta.dimensions[0]}x{meta.dimensions[1]}, "
      f"{meta.level_count} levels, ds={meta.level_downsamples}")
v

## 2. See the cross-fade

In the canvas view, quickly scroll-zoom from the whole-slide view (a
coarse level) down to high magnification (level 0) — e.g. double-click
to zoom in twice in a row, or drag the minimap. As the selected pyramid
level changes, the *old* level does not pop off: it fades out over
`BLEND_MS = 300 ms` while the new level draws at full opacity
(`frontend/blend.js`). The fade is computed in the compositor per level
(`levelAlphas`), coarsest level first, so the fine level is always on top
while the coarse one dissolves underneath.

Note what does **not** happen: the canvas never blanks between the two
pushes of a two-stage render (the view draws everything accumulated in
`_tileGeo`, not just the latest push), and at rest there is no animation
loop — the rAF stepping stops when `blend.done(...)` turns true.


## 3. The chunked read plan (pure math, no pixels)

`plan_viewport` now carries the tile set **and** its chunking: 1024-px
blocks, grid-anchored, center-first. At a wide viewport the plan spans
several blocks; the order below is the fetch order — and, for a
multi-chunk render, the first block is the one pushed before the rest.

In [ ]:
from islide.plan import plan_viewport
from islide.viewport import Viewport

cx, cy = meta.dimensions[0] / 2, meta.dimensions[1] / 2


def show_plan(zoom: float, dx: float = 0, dy: float = 0) -> None:
    vp = Viewport(cx + dx, cy + dy, zoom, canvas_w=960, canvas_h=540)
    plan = plan_viewport(meta, vp, tile_size=256)
    print(f"zoom {zoom:>6}: level {plan.level}, "
          f"{len(plan.tiles)} tiles in {len(plan.chunks)} chunk(s)")
    for i, c in enumerate(plan.chunks):
        print(f"  chunk {i}: origin {c.read_origin}, size {c.size}, "
              f"{len(c.tiles)} tiles")


# wide view: the canvas spans several 1024-px blocks -> multi-chunk plan
show_plan(0.5)
# tight view: one block -> the pre-M5 fast path (one read, one push)
# (offset 100 px from the exact center so the view sits inside one
# 1024-px block instead of on its corner)
show_plan(16.0, dx=100, dy=100)

## 4. Watch the two-stage push

Set up an observer on `tiles` before zooming, then change the viewport
in the view (or via `v.set_zoom`). A wide, multi-chunk viewport produces
**two** `tiles` pushes — the center chunk first, then the full set — while
a single-chunk viewport produces exactly one. Both pushes carry the same
absolute `tile_geo` geometry; only the set of keys grows.

In [ ]:
pushes = []
v.observe(lambda c: pushes.append(c["new"]), names="tiles")

# wide view (multi-chunk): expect two pushes, first is the center chunk
v.set_zoom(0.5, cx=meta.dimensions[0] / 2, cy=meta.dimensions[1] / 2)
print(f"pushes after wide view: {len(pushes)}")
if len(pushes) >= 2:
    print(f"  first push: {len(pushes[0])} tiles (center chunk)")
    print(f"  full set:   {len(pushes[-1])} tiles")

# tight view (single chunk): expect exactly one push, one read
# (offset 100 px from the exact center so the view is well inside
# one 1024-px block, not on a block boundary)
pushes.clear()
v.set_zoom(16.0, cx=meta.dimensions[0] / 2 + 100, cy=meta.dimensions[1] / 2 + 100)
print(f"pushes after tight view: {len(pushes)} "
      f"({len(pushes[0]) if pushes else 0} tiles)")

## 5. What this is not

- Not multi-level *serving*: the view still shows one level at a time;
  the second level is only visible *during* the 300 ms fade.
- Not a tile cache change: tiles are cached by key as before; chunks
  only change how reads are grouped and in which order they arrive.
- Not a wire change: `tiles`, `tile_geo`, `minimap_img`, the overlay
  traits, and the annotation contract are exactly as in M4; the JS module
  version is still 2.0.0.
